# Inference Speed Test (YOLO Eye Classifier)

This notebook benchmarks inference latency/FPS for available YOLO classification weights using real eye images from the local `blinkblink-*` datasets.

It measures:
- **Preprocess time** (`preprocess_for_yolo`)
- **Model inference time** (`YOLO(preprocessed)`)
- **End-to-end time** (preprocess + inference)

> Notes:
> - First runs are slower due to initialization; warmup iterations are excluded.
> - GPU is used automatically if available and `USE_GPU = True`.

In [ ]:
from pathlib import Path
import random
import time
from statistics import mean, median

import cv2
import numpy as np
import pandas as pd
import torch
from ultralytics import YOLO
from IPython.display import display


def preprocess_for_yolo(image: np.ndarray) -> np.ndarray:
    if image is None or image.size == 0:
        return image

    img = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (512, 512), interpolation=cv2.INTER_LINEAR)
    lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    l = cv2.equalizeHist(l)
    img = cv2.cvtColor(cv2.merge((l, a, b)), cv2.COLOR_LAB2RGB)
    return img


# Reproducibility
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# Benchmark settings
USE_GPU = True
NUM_IMAGES = 300        # sampled images per model
WARMUP_ITERS = 30       # excluded from timing stats
TIMED_ITERS = 300       # measured forward passes per model

# Input discovery settings
DATASET_GLOB = "blinkblink-*/**/*"
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

# Candidate model paths (ordered)
MODEL_CANDIDATES = [
    "runs/classify/nano_100/weights/best.pt",
    "runs/classify/small_100/weights/best.pt",
    "runs/classify/medium_100/weights/best.pt",
    "runs/classify/nano_100_aug/weights/best.pt",
    "runs/classify/small_100_aug/weights/best.pt",
    "runs/classify/medium_100_aug/weights/best.pt",
]

ROOT = Path.cwd()
DEVICE = "cuda" if (USE_GPU and torch.cuda.is_available()) else "cpu"
print(f"Using device: {DEVICE}")

Using device: cuda


In [3]:
def find_available_models(root: Path, candidates: list[str]) -> list[Path]:
    models = []
    for rel in candidates:
        p = root / rel
        if p.exists() and p.is_file():
            models.append(p)
    return models


def find_images(root: Path, glob_pattern: str, suffixes: set[str]) -> list[Path]:
    images = []
    for p in root.glob(glob_pattern):
        if p.is_file() and p.suffix.lower() in suffixes:
            images.append(p)
    return images


def load_and_sample_images(image_paths: list[Path], n: int) -> list[np.ndarray]:
    if not image_paths:
        return []

    if len(image_paths) > n:
        selected_paths = random.sample(image_paths, n)
    else:
        selected_paths = image_paths

    frames = []
    for p in selected_paths:
        img = cv2.imread(str(p))
        if img is not None and img.size > 0:
            frames.append(img)
    return frames


def percentile(values: list[float], q: float) -> float:
    # q in [0, 100]
    if not values:
        return float("nan")
    return float(np.percentile(np.asarray(values, dtype=np.float64), q))


model_paths = find_available_models(ROOT, MODEL_CANDIDATES)
all_images = find_images(ROOT, DATASET_GLOB, IMAGE_SUFFIXES)
sample_images = load_and_sample_images(all_images, NUM_IMAGES)

print(f"Discovered models: {len(model_paths)}")
for m in model_paths:
    print(f"  - {m}")

print(f"Discovered images: {len(all_images)}")
print(f"Loaded sample images: {len(sample_images)}")

if not model_paths:
    raise RuntimeError("No model weights found. Update MODEL_CANDIDATES.")
if not sample_images:
    raise RuntimeError("No dataset images found. Check DATASET_GLOB / IMAGE_SUFFIXES.")

Discovered models: 9
  - c:\Gojiii\blink blink\repo\eye-blink-decoder\yolo26n-cls.pt
  - c:\Gojiii\blink blink\repo\eye-blink-decoder\yolo26s-cls.pt
  - c:\Gojiii\blink blink\repo\eye-blink-decoder\yolo26m-cls.pt
  - c:\Gojiii\blink blink\repo\eye-blink-decoder\runs\classify\nano_100\weights\best.pt
  - c:\Gojiii\blink blink\repo\eye-blink-decoder\runs\classify\small_100\weights\best.pt
  - c:\Gojiii\blink blink\repo\eye-blink-decoder\runs\classify\medium_100\weights\best.pt
  - c:\Gojiii\blink blink\repo\eye-blink-decoder\runs\classify\nano_100_aug\weights\best.pt
  - c:\Gojiii\blink blink\repo\eye-blink-decoder\runs\classify\small_100_aug\weights\best.pt
  - c:\Gojiii\blink blink\repo\eye-blink-decoder\runs\classify\medium_100_aug\weights\best.pt
Discovered images: 12912
Loaded sample images: 300


In [4]:
def benchmark_model(model_path: Path, images: list[np.ndarray], device: str,
                    warmup_iters: int, timed_iters: int) -> dict:
    model = YOLO(str(model_path))
    if device == "cuda":
        model.to("cuda")

    preprocess_times = []
    infer_times = []
    e2e_times = []

    # Total iterations can exceed image count; reuse images cyclically
    total_iters = warmup_iters + timed_iters

    for i in range(total_iters):
        img = images[i % len(images)]

        t0 = time.perf_counter()
        pre = preprocess_for_yolo(img)
        t1 = time.perf_counter()

        _ = model(pre, verbose=False)

        if device == "cuda":
            torch.cuda.synchronize()

        t2 = time.perf_counter()

        if i >= warmup_iters:
            pre_ms = (t1 - t0) * 1000.0
            infer_ms = (t2 - t1) * 1000.0
            e2e_ms = (t2 - t0) * 1000.0

            preprocess_times.append(pre_ms)
            infer_times.append(infer_ms)
            e2e_times.append(e2e_ms)

    return {
        "model": str(model_path),
        "device": device,
        "samples": timed_iters,
        "pre_mean_ms": mean(preprocess_times),
        "pre_median_ms": median(preprocess_times),
        "pre_p95_ms": percentile(preprocess_times, 95),
        "infer_mean_ms": mean(infer_times),
        "infer_median_ms": median(infer_times),
        "infer_p95_ms": percentile(infer_times, 95),
        "e2e_mean_ms": mean(e2e_times),
        "e2e_median_ms": median(e2e_times),
        "e2e_p95_ms": percentile(e2e_times, 95),
        "fps_mean": 1000.0 / mean(e2e_times),
        "fps_median": 1000.0 / median(e2e_times),
    }


rows = []
for idx, model_path in enumerate(model_paths, start=1):
    print(f"[{idx}/{len(model_paths)}] Benchmarking: {model_path.name}")
    row = benchmark_model(
        model_path=model_path,
        images=sample_images,
        device=DEVICE,
        warmup_iters=WARMUP_ITERS,
        timed_iters=TIMED_ITERS,
    )
    rows.append(row)

results_df = pd.DataFrame(rows)
results_df = results_df.sort_values(by="e2e_mean_ms", ascending=True).reset_index(drop=True)

display(results_df)

best = results_df.iloc[0]
print("\nFastest model by mean end-to-end latency:")
print(f"  Model: {Path(best['model']).name}")
print(f"  e2e_mean_ms: {best['e2e_mean_ms']:.3f}")
print(f"  fps_mean: {best['fps_mean']:.2f}")

[1/9] Benchmarking: yolo26n-cls.pt
[2/9] Benchmarking: yolo26s-cls.pt
[3/9] Benchmarking: yolo26m-cls.pt
[4/9] Benchmarking: best.pt
[5/9] Benchmarking: best.pt
[6/9] Benchmarking: best.pt
[7/9] Benchmarking: best.pt
[8/9] Benchmarking: best.pt
[9/9] Benchmarking: best.pt


,model,device,samples,pre_mean_ms,pre_median_ms,pre_p95_ms,infer_mean_ms,infer_median_ms,infer_p95_ms,e2e_mean_ms,e2e_median_ms,e2e_p95_ms,fps_mean,fps_median
0,c:\Gojiii\blink blink\repo\eye-blink-decoder\r...,cuda,300,3.857206,3.32740,5.078210,9.405307,8.81080,11.558435,13.262513,12.46235,16.548340,75.400490,80.241688
1,c:\Gojiii\blink blink\repo\eye-blink-decoder\r...,cuda,300,3.941626,3.37095,5.239320,9.514082,9.18660,11.483330,13.455709,12.87845,16.565035,74.317899,77.649096
2,c:\Gojiii\blink blink\repo\eye-blink-decoder\r...,cuda,300,3.813776,3.31125,5.075710,9.720409,9.03995,11.786865,13.534185,12.57805,16.848675,73.886976,79.503580
3,c:\Gojiii\blink blink\repo\eye-blink-decoder\y...,cuda,300,4.042559,3.58500,5.173990,9.764059,9.47130,11.959510,13.806618,13.64130,16.913700,72.429035,73.306796
4,c:\Gojiii\blink blink\repo\eye-blink-decoder\y...,cuda,300,3.938533,3.37515,5.047000,9.916165,9.47790,11.856365,13.854698,13.40650,16.812540,72.177683,74.590684
5,c:\Gojiii\blink blink\repo\eye-blink-decoder\r...,cuda,300,4.119646,3.57210,5.636970,10.558461,9.91295,12.732120,14.678107,13.64265,18.279145,68.128676,73.299542
6,c:\Gojiii\blink blink\repo\eye-blink-decoder\r...,cuda,300,3.868293,3.31620,5.157875,11.370073,10.52750,14.043675,15.238367,14.24460,18.936475,65.623831,70.202041
7,c:\Gojiii\blink blink\repo\eye-blink-decoder\r...,cuda,300,4.006262,3.51480,5.553425,11.752479,11.08225,14.831490,15.758742,14.82220,20.214550,63.456843,67.466368
8,c:\Gojiii\blink blink\repo\eye-blink-decoder\y...,cuda,300,4.003046,3.42985,5.172005,11.924452,11.49905,15.056475,15.927498,15.41135,20.241960,62.784500,64.887242



Fastest model by mean end-to-end latency:
  Model: best.pt
  e2e_mean_ms: 13.263
  fps_mean: 75.40


In [5]:
# Optional: save benchmark report
out_path = ROOT / "runs" / "classify" / "val" / "inference_speed_results.csv"
out_path.parent.mkdir(parents=True, exist_ok=True)
results_df.to_csv(out_path, index=False)
print(f"Saved: {out_path}")

Saved: c:\Gojiii\blink blink\repo\eye-blink-decoder\runs\classify\val\inference_speed_results.csv
